# LoRA Fine-Tuning

In this notebook we will implement [Low-Rank Adaptation, LoRA](https://arxiv.org/abs/2106.09685), as an approach for fine-tuning.

This notebook is inspired by Appendix E in of [Build A Larger Language Model (from scratch)](https://sebastianraschka.com/llms-from-scratch/) by Sebastian Raschka and I highly recommend the book.

In fine-tuning the goal is to modify the parameters, $W_0$, of a LLM so that the updated parameters $W = W_0 + \Delta W$ improves the performance of the LLM on a given task. 
A popular approach for fine-tuning large language models is LoRA where instead of optimizing all parameters, a low-rank adaption is used:

$$
 W = W_0 + \Delta W = W_0 + BA
$$

where $A \in \R^{r \times k} $, $B \in \R^{d \times r}$ and $W_0 \in \R ^{d \times k}$ with $r \ll \min(d, k)$. So instead of optimizing $\Delta W$, we optimize $A$ and $B$ which contain significantly fewer parameters.

Below we implement the LoRA adaptation with $r$ represented by `rank`. A scaling hyperparameter `alpha` is introduced which tunes the size of the effect of the LoRA adaption on the original model.

In [8]:
import torch

class LoRALayer(torch.nn.Module):
    def __init__(self, d_in, d_out, rank, alpha):
        self.A = torch.nn.Parameter(torch.empty(d_in, rank))
        torch.nn.init.kaiming_uniform_(self.A, a=torch.sqrt(torch.tensor(5)))
        self.B = torch.nn.Parameter(torch.zeros(rank, d_out))
        self.alpha = alpha
        self.rank = rank

    def forward(self, x):
        x = (self.alpha / self.rank) * (x @ self.A @ self.B)
        return x


While the original paper mentions a Gaussian initialization for $A$, current practize seems to be using a uniform [Kaiming](https://arxiv.org/abs/1502.01852) initialization for $A$ so we will stick with that. On the other hand, $B$ is initialized to zero. In that way, the fine-tuning will start with a model that is effectively the same as the original model.

Let's make a torch module for a linear layer with LoRA adaptation:

In [10]:
class LinearWithLoRA(torch.nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.lora = LoRALayer(linear.in_features, linear.out_features, rank, alpha)
        self.linear = linear

        return

    def forward(self, x):
        return self.linear(x) + self.lora(x)

As usual we will instantiate our implementation of GPT2 and load the OpenAI weights from Hugging Face:

In [11]:
from sturnus.model import GPTModel

GPT_CONFIG_124_openai = {
    'vocab_size': 50257,
    'block_size': 1024,
    'count_heads': 12,
    'count_blocks': 12,
    'embed_dim': 768,
    'dropout': 0.1,
    'qkv_bias': True, # Used in GPT2 but typically not in modern LLMs as the biases do not improve performance
}

model = GPTModel(GPT_CONFIG_124_openai)